In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer , IterativeImputer
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline 
from sklearn.model_selection import cross_val_score

: 

In [3]:
df = pd.read_csv("titanic.csv")

In [25]:

# df.drop("deck",axis=1,inplace=True)
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive',
       'alone'],
      dtype='object')

In [40]:
X = df.drop("survived",axis=1)
y = df["survived"]

num_features = ['age','fare']
cat_features = ['pclass','sex','embarked']

num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())])

cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ("num",num_pipeline, num_features),
    ("cat",cat_pipeline, cat_features)
])

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model" , LogisticRegression(max_iter=1000 , random_state=42))
])

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=42)

pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)

print("Accuracy:",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

scores = cross_val_score(pipeline,X,y,cv=5)
print("CV Accuracy:",scores.mean())


Accuracy: 0.7947761194029851
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       157
           1       0.78      0.70      0.74       111

    accuracy                           0.79       268
   macro avg       0.79      0.78      0.79       268
weighted avg       0.79      0.79      0.79       268

CV Accuracy: 0.7890088506685079


In [44]:
df.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
embark_town      2
alive            0
alone            0
dtype: int64

In [51]:
X = df.drop("survived",axis=1)
y = df['survived']

num_features = ['age','fare']
cat_features = ['sex','embarked']

num_pipeline = Pipeline([
    ("imputer",IterativeImputer(max_iter=100,random_state=42)),
    ("scaler",StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num",num_pipeline,num_features),
    ("cat",cat_pipeline,cat_features)
])

pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30,random_state=42)

pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)